In [ ]:
import pandas as pd
import tensorflow as tf
import numpy as np

In [ ]:
tweets=pd.read_csv('/content/imdb_10K_sentimnets_reviews.csv')

In [ ]:
tweets.head()

In [ ]:
len(tweets[tweets['sentiment']==1]),len(tweets[tweets['sentiment']==0])

In [ ]:
import nltk 
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import TweetTokenizer
import string 
import re
     

In [ ]:
import nltk
nltk.download('stopwords')

In [ ]:
#process tweets
def process_tweet(tweet):
    stemmer=PorterStemmer()
    stopwords_english = stopwords.words('english') #stopwords english
    
    #removing all hashtags ,hyperlinks
    tweet = re.sub(r'$\w*', '', tweet)
    tweet = re.sub(r'^RT[\s]+', '', tweet)
    tweet = re.sub(r'https?:\/\/.*[\r\n]*', '', tweet)
    tweet = re.sub(r'#', '', tweet)
    
    tokenizer=TweetTokenizer(preserve_case=False, strip_handles=True,reduce_len=True)
    tweet_tokens=tokenizer.tokenize(tweet) #tokenize
    
    tweets_clean=[]
    
    #removing stopwords, removing punctuation and then stemming the word
    for word in tweet_tokens:
        if (word not in stopwords_english and word not in string.punctuation):   
            stem_word=stemmer.stem(word)
            tweets_clean.append(stem_word)
            
    return tweets_clean

In [ ]:
from keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM


In [ ]:

#creating vocablory 
vocab=[]
for tweet in tweets.review:
  for word in process_tweet(tweet):
    if word not in vocab:
      vocab.append(word)


In [ ]:
print("length of vocab:",len(vocab))

In [ ]:

#process tweet
processed_tweet=[]
for t in tweets.review:
  p_t=process_tweet(t)
  processed_tweet.append(p_t)

In [ ]:
tweets['p_tweet']=processed_tweet
tweets

In [ ]:
tokenizer = Tokenizer(num_words=len(vocab), split=' ') 
tokenizer.fit_on_texts(tweets['p_tweet'].values)
X = tokenizer.texts_to_sequences(tweets['p_tweet'])
X=pad_sequences(X)


In [ ]:
len(X[0])

In [ ]:
Y=tweets['sentiment'].to_list()
Y=np.array(Y)

In [ ]:
embeding_dim=128
lstm_out=196
model = Sequential()
model.add(Embedding(input_dim=len(vocab),output_dim=embeding_dim,input_length = 1185))
model.add(LSTM(lstm_out, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
model.compile(loss = 'binary_crossentropy', optimizer='adam',metrics = ['accuracy'])

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size = 0.15, random_state = 42)

In [ ]:
model.fit(X_train, y_train,validation_data = (X_test,y_test),epochs = 1, batch_size=32)

In [ ]:
model.save('movie_sentiment_model.h5')

In [ ]:
test_tweet=tweets['review'][0]

In [ ]:
test_tweet

In [ ]:
processed_tweet=process_tweet(test_tweet)
processed_tweet

In [ ]:
Pro_seq_tweet=tokenizer.texts_to_sequences(processed_tweet)
Pro_seq_tweet

In [ ]:
input_review=[]
for i in Pro_seq_tweet:
  input_review.append(i[0])
input_review

In [ ]:
input_review1=pad_sequences([input_review],maxlen=1185)

In [ ]:
model.predict(input_review1)

In [ ]:
def review_list(review):
  input_review=[]
  for i in review:
    input_review.append(i[0])
  return input_review

def pos_neg(pred):
  if pred>0.5:
    return "positive review"
  else:
    return "negative review"

In [ ]:
review="The Movie was good, i loved it!!"
pro_review=process_tweet(review)
pro_seq_review=tokenizer.texts_to_sequences(pro_review)
input_review=review_list(pro_seq_review)
pad_pro_seq=pad_sequences([input_review],maxlen=1185)


pred=model.predict(pad_pro_seq)
print("Prediction score",pred[0][0])

print()
print()
print("Review: ",review)
print("Predicted review: ",pos_neg(pred[0][0]))


In [ ]:
review="The Movie was bad, it was a waste of time!!"
pro_review=process_tweet(review)
pro_seq_review=tokenizer.texts_to_sequences(pro_review)
input_review=review_list(pro_seq_review)
pad_pro_seq=pad_sequences([input_review],maxlen=1185)


pred=model.predict(pad_pro_seq)
print("Prediction score",pred[0][0])

print()
print()
print("Review: ",review)
print("Predicted review: ",pos_neg(pred[0][0]))


model prediction for top 6 test  tweets

In [ ]:
model.predict(X_test[0:6])